# MLB data grab and local export notebook

This notebook is a cloud-runtime data grabber. Run it in Vertex AI Workbench, Colab, Cloud Shell Jupyter, or any environment that can reach the MLB/Statcast/Odds APIs. It collects raw data, builds the feature dataframe inline, then packages parquet/CSV outputs so you can download them to your local PC and do EDA/modeling without repeatedly paying for Vertex runtime.

Main output: `mlb_data_export/processed/mlb_game_features.parquet`.

Optional raw outputs: schedule, team boxscores, pitcher boxscores, odds snapshots, and a Statcast sample/schema. Set `EXPORT_RAW_STATCAST=True` if you want to download the full pitch-level Statcast table too.


In [ ]:

# Uncomment this in a fresh Colab / Vertex AI Workbench runtime if needed.
# %pip install -q pandas numpy scipy requests pyarrow pybaseball tqdm

from __future__ import annotations

import json
import math
import os
import shutil
import time
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)

print("Setup complete")
print("pandas", pd.__version__)


In [ ]:

# -----------------------
# Data grab config
# -----------------------
PROJECT_ROOT = Path.cwd()
EXPORT_ROOT = PROJECT_ROOT / "mlb_data_export"
RAW_DIR = EXPORT_ROOT / "raw"
PROCESSED_DIR = EXPORT_ROOT / "processed"
REPORTS_DIR = EXPORT_ROOT / "reports"
for p in [EXPORT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TODAY = pd.Timestamp.utcnow().date()

# Start with a regular-season date range. You can widen this when needed.
START_DATE = os.getenv("START_DATE", "2023-01-01")
END_DATE = os.getenv("END_DATE", (TODAY + timedelta(days=2)).isoformat())
GAME_TYPE = os.getenv("MLB_GAME_TYPE", "R")  # R = regular season

SPORT_KEY = os.getenv("ODDS_SPORT_KEY", "baseball_mlb")
ODDS_REGIONS = os.getenv("ODDS_REGIONS", "us")
ODDS_MARKETS = os.getenv("ODDS_MARKETS", "h2h,spreads,totals")
ODDS_FORMAT = os.getenv("ODDS_FORMAT", "american")
ODDS_API_KEY = os.getenv("ODDS_API_KEY", "")  # Optional. Leave blank to skip odds.

# Toggle each external API. This notebook is intended to run in a cloud notebook runtime.
FETCH_SCHEDULE = True
FETCH_BOXSCORES = True
FETCH_ODDS = bool(ODDS_API_KEY)
FETCH_STATCAST = True

# Statcast can be slow/large. Use a smaller chunk if Baseball Savant is flaky.
SCHEDULE_CHUNK_DAYS = 30
STATCAST_CHUNK_DAYS = 7
API_SLEEP_SECONDS = 0.15

# Export options.
EXPORT_RAW_TABLES = True
EXPORT_RAW_STATCAST = False  # Set True if you want the huge pitch-level raw table on your PC.
EXPORT_CSV_SAMPLE = True
MAKE_ZIP = True
ZIP_NAME = f"mlb_data_export_{START_DATE}_to_{END_DATE}.zip".replace(":", "-")

print({
    "START_DATE": START_DATE,
    "END_DATE": END_DATE,
    "GAME_TYPE": GAME_TYPE,
    "FETCH_SCHEDULE": FETCH_SCHEDULE,
    "FETCH_BOXSCORES": FETCH_BOXSCORES,
    "FETCH_ODDS": FETCH_ODDS,
    "FETCH_STATCAST": FETCH_STATCAST,
    "EXPORT_ROOT": str(EXPORT_ROOT),
})


In [ ]:

def normalize_team_name(x):
    if x is None or pd.isna(x):
        return ""

    s = str(x).strip().lower()
    s = s.replace(".", "")
    s = s.replace("'", "")
    s = s.replace("&", "and")
    s = " ".join(s.split())

    aliases = {
        "athletics": "oakland athletics",
        "the athletics": "oakland athletics",
        "oakland athletics": "oakland athletics",
        "as": "oakland athletics",
        "a's": "oakland athletics",
        "ath": "oakland athletics",
        "oak": "oakland athletics",

        "ari": "arizona diamondbacks",
        "az": "arizona diamondbacks",
        "atl": "atlanta braves",
        "bal": "baltimore orioles",
        "bos": "boston red sox",
        "chc": "chicago cubs",
        "chw": "chicago white sox",
        "cws": "chicago white sox",
        "cin": "cincinnati reds",
        "cle": "cleveland guardians",
        "col": "colorado rockies",
        "det": "detroit tigers",
        "hou": "houston astros",
        "kc": "kansas city royals",
        "kcr": "kansas city royals",
        "laa": "los angeles angels",
        "lad": "los angeles dodgers",
        "mia": "miami marlins",
        "mil": "milwaukee brewers",
        "min": "minnesota twins",
        "nym": "new york mets",
        "nyy": "new york yankees",
        "phi": "philadelphia phillies",
        "pit": "pittsburgh pirates",
        "sd": "san diego padres",
        "sdp": "san diego padres",
        "sf": "san francisco giants",
        "sfg": "san francisco giants",
        "sea": "seattle mariners",
        "stl": "st louis cardinals",
        "tb": "tampa bay rays",
        "tbr": "tampa bay rays",
        "tex": "texas rangers",
        "tor": "toronto blue jays",
        "wsh": "washington nationals",
        "was": "washington nationals",
        "wsn": "washington nationals",
    }

    return aliases.get(s, s)


def american_to_implied_prob(price: Any) -> float:
    if price is None or pd.isna(price):
        return np.nan
    p = float(price)
    if p > 0:
        return 100.0 / (p + 100.0)
    return abs(p) / (abs(p) + 100.0)


def american_profit_per_unit(price: Any) -> float:
    if price is None or pd.isna(price):
        return np.nan
    p = float(price)
    if p > 0:
        return p / 100.0
    return 100.0 / abs(p)


def expected_value_per_unit(model_prob: float, american_price: float) -> float:
    if pd.isna(model_prob) or pd.isna(american_price):
        return np.nan
    profit = american_profit_per_unit(american_price)
    return model_prob * profit - (1.0 - model_prob)


def no_vig_two_way_prob(price_a: Any, price_b: Any) -> tuple[float, float]:
    pa = american_to_implied_prob(price_a)
    pb = american_to_implied_prob(price_b)
    if pd.isna(pa) or pd.isna(pb) or (pa + pb) <= 0:
        return np.nan, np.nan
    return pa / (pa + pb), pb / (pa + pb)


def daterange_chunks(start_date: str | date, end_date: str | date, chunk_days: int):
    start = pd.to_datetime(start_date).date()
    end = pd.to_datetime(end_date).date()
    cur = start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        yield cur, chunk_end
        cur = chunk_end + timedelta(days=1)


def safe_to_parquet(df: pd.DataFrame, path: str | Path, **kwargs) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False, **kwargs)


def show_df(name: str, df: pd.DataFrame, n: int = 5) -> None:
    print(f"{name}: shape={getattr(df, 'shape', None)}")
    if isinstance(df, pd.DataFrame) and not df.empty:
        display(df.head(n))


In [ ]:

def fetch_mlb_schedule(start_date: str, end_date: str, game_type: str = "R", chunk_days: int = 30) -> pd.DataFrame:
    """Fetch MLB games from the public MLB Stats API schedule endpoint."""
    rows = []
    for s, e in daterange_chunks(start_date, end_date, chunk_days):
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {
            "sportId": 1,
            "startDate": s.isoformat(),
            "endDate": e.isoformat(),
            "gameTypes": game_type,
            "hydrate": "probablePitcher,linescore",
        }
        r = requests.get(url, params=params, timeout=45)
        r.raise_for_status()
        data = r.json()
        for dblock in data.get("dates", []) or []:
            for game in dblock.get("games", []) or []:
                teams = game.get("teams", {}) or {}
                home = teams.get("home", {}) or {}
                away = teams.get("away", {}) or {}
                home_team = (home.get("team", {}) or {})
                away_team = (away.get("team", {}) or {})
                status = game.get("status", {}) or {}
                hp = home.get("probablePitcher", {}) or {}
                ap = away.get("probablePitcher", {}) or {}
                rows.append({
                    "game_pk": game.get("gamePk"),
                    "official_date": game.get("officialDate") or dblock.get("date"),
                    "game_datetime_utc": game.get("gameDate"),
                    "game_type": game.get("gameType"),
                    "detailed_state": status.get("detailedState"),
                    "abstract_state": status.get("abstractGameState"),
                    "home_team_id": home_team.get("id"),
                    "home_team_name": home_team.get("name"),
                    "home_team_norm": normalize_team_name(home_team.get("name")),
                    "away_team_id": away_team.get("id"),
                    "away_team_name": away_team.get("name"),
                    "away_team_norm": normalize_team_name(away_team.get("name")),
                    "home_score": home.get("score"),
                    "away_score": away.get("score"),
                    "home_probable_pitcher_id": hp.get("id"),
                    "home_probable_pitcher_name": hp.get("fullName"),
                    "away_probable_pitcher_id": ap.get("id"),
                    "away_probable_pitcher_name": ap.get("fullName"),
                })
        print(f"schedule {s} to {e}: cumulative rows={len(rows)}")
        time.sleep(API_SLEEP_SECONDS)

    df = pd.DataFrame(rows).drop_duplicates("game_pk", keep="last")
    if not df.empty:
        df["official_date"] = pd.to_datetime(df["official_date"], errors="coerce")
        df["game_datetime_utc"] = pd.to_datetime(df["game_datetime_utc"], errors="coerce", utc=True)
        for c in ["game_pk", "home_score", "away_score", "home_team_id", "away_team_id", "home_probable_pitcher_id", "away_probable_pitcher_id"]:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        df["home_team_norm"] = df["home_team_name"].apply(normalize_team_name)
        df["away_team_norm"] = df["away_team_name"].apply(normalize_team_name)
        df["is_final"] = df["abstract_state"].eq("Final") | df["detailed_state"].astype(str).str.lower().isin(["final", "completed early"])
        df["target_home_win"] = np.where(
            df["is_final"] & df["home_score"].notna() & df["away_score"].notna(),
            (df["home_score"] > df["away_score"]).astype(float),
            np.nan,
        )
        df["target_total_runs"] = np.where(df["is_final"], df["home_score"] + df["away_score"], np.nan)
        df["target_home_margin"] = np.where(df["is_final"], df["home_score"] - df["away_score"], np.nan)
    return df


def fetch_mlb_boxscore(game_pk: int) -> dict:
    url = f"https://statsapi.mlb.com/api/v1/game/{int(game_pk)}/boxscore"
    r = requests.get(url, timeout=45)
    r.raise_for_status()
    return r.json()


def innings_to_float(value: Any) -> float:
    """Convert MLB innings string like '5.2' to 5 + 2/3."""
    if value is None or pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s:
        return np.nan
    try:
        if "." in s:
            whole, outs = s.split(".", 1)
            return float(whole) + float(outs) / 3.0
        return float(s)
    except Exception:
        return np.nan


def parse_boxscore_team_rows(game_row: pd.Series, box: dict) -> list[dict]:
    out = []
    teams = box.get("teams", {}) or {}
    for side in ["home", "away"]:
        t = teams.get(side, {}) or {}
        team_meta = t.get("team", {}) or {}
        batting = ((t.get("teamStats", {}) or {}).get("batting", {}) or {})
        pitching = ((t.get("teamStats", {}) or {}).get("pitching", {}) or {})
        row = {
            "game_pk": game_row.get("game_pk"),
            "official_date": game_row.get("official_date"),
            "game_datetime_utc": game_row.get("game_datetime_utc"),
            "team_side": side,
            "team_id": team_meta.get("id") or game_row.get(f"{side}_team_id"),
            "team_name": team_meta.get("name") or game_row.get(f"{side}_team_name"),
            "team_norm": normalize_team_name(team_meta.get("name") or game_row.get(f"{side}_team_name")),
            "opponent_team_id": game_row.get("away_team_id" if side == "home" else "home_team_id"),
            "opponent_team_name": game_row.get("away_team_name" if side == "home" else "home_team_name"),
            "runs_for": game_row.get("home_score" if side == "home" else "away_score"),
            "runs_against": game_row.get("away_score" if side == "home" else "home_score"),
        }
        for k, v in batting.items():
            row[f"box_bat_{k}"] = v
        for k, v in pitching.items():
            row[f"box_pitch_{k}"] = v
        out.append(row)
    return out


def parse_boxscore_pitcher_rows(game_row: pd.Series, box: dict) -> list[dict]:
    """Parse pitcher-game boxscore rows, including starter flags and per-game ERA/WHIP."""
    out = []
    teams = box.get("teams", {}) or {}
    for side in ["home", "away"]:
        t = teams.get(side, {}) or {}
        team_meta = t.get("team", {}) or {}
        players = t.get("players", {}) or {}
        pitcher_order = t.get("pitchers", []) or []
        starter_id = int(pitcher_order[0]) if pitcher_order else None
        if starter_id is None and pd.notna(game_row.get(f"{side}_probable_pitcher_id")):
            starter_id = int(game_row.get(f"{side}_probable_pitcher_id"))

        for player_key, pinfo in players.items():
            person = pinfo.get("person", {}) or {}
            pid = person.get("id")
            stats = ((pinfo.get("stats", {}) or {}).get("pitching", {}) or {})
            if not stats:
                continue
            ip = innings_to_float(stats.get("inningsPitched"))
            hits = pd.to_numeric(stats.get("hits"), errors="coerce")
            er = pd.to_numeric(stats.get("earnedRuns"), errors="coerce")
            bb = pd.to_numeric(stats.get("baseOnBalls"), errors="coerce")
            so = pd.to_numeric(stats.get("strikeOuts"), errors="coerce")
            hr = pd.to_numeric(stats.get("homeRuns"), errors="coerce")
            pitches = pd.to_numeric(stats.get("pitchesThrown"), errors="coerce")

            row = {
                "game_pk": game_row.get("game_pk"),
                "official_date": game_row.get("official_date"),
                "game_datetime_utc": game_row.get("game_datetime_utc"),
                "team_side": side,
                "team_id": team_meta.get("id") or game_row.get(f"{side}_team_id"),
                "team_name": team_meta.get("name") or game_row.get(f"{side}_team_name"),
                "team_norm": normalize_team_name(team_meta.get("name") or game_row.get(f"{side}_team_name")),
                "pitcher_id": pid,
                "pitcher_name": person.get("fullName"),
                "is_starting_pitcher": int(pid == starter_id) if pid is not None and starter_id is not None else 0,
                "p_ip_float": ip,
                "p_hits": hits,
                "p_runs": pd.to_numeric(stats.get("runs"), errors="coerce"),
                "p_earned_runs": er,
                "p_base_on_balls": bb,
                "p_strikeouts": so,
                "p_home_runs": hr,
                "p_pitches_thrown": pitches,
                "p_game_era": (er * 9.0 / ip) if pd.notna(ip) and ip > 0 and pd.notna(er) else np.nan,
                "p_game_whip": ((hits + bb) / ip) if pd.notna(ip) and ip > 0 and pd.notna(hits) and pd.notna(bb) else np.nan,
                "p_k_per_9": (so * 9.0 / ip) if pd.notna(ip) and ip > 0 and pd.notna(so) else np.nan,
                "p_bb_per_9": (bb * 9.0 / ip) if pd.notna(ip) and ip > 0 and pd.notna(bb) else np.nan,
                "p_hr_per_9": (hr * 9.0 / ip) if pd.notna(ip) and ip > 0 and pd.notna(hr) else np.nan,
            }
            for k, v in stats.items():
                row[f"pitch_box_{k}"] = v
            out.append(row)
    return out


def fetch_boxscores_for_games_full(games_df: pd.DataFrame, only_final: bool = True, limit: int | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    candidates = games_df.copy()
    if only_final and "is_final" in candidates.columns:
        candidates = candidates[candidates["is_final"].eq(True)]
    candidates = candidates.sort_values(["official_date", "game_datetime_utc", "game_pk"])
    if limit:
        candidates = candidates.head(limit)

    team_rows = []
    pitcher_rows = []
    for i, (_, g) in enumerate(candidates.iterrows(), start=1):
        try:
            box = fetch_mlb_boxscore(int(g["game_pk"]))
            team_rows.extend(parse_boxscore_team_rows(g, box))
            pitcher_rows.extend(parse_boxscore_pitcher_rows(g, box))
        except Exception as exc:
            print(f"boxscore failed game_pk={g.get('game_pk')}: {exc}")
        if i % 100 == 0:
            print(f"boxscores processed {i}/{len(candidates)}")
        time.sleep(API_SLEEP_SECONDS)
    return pd.DataFrame(team_rows), pd.DataFrame(pitcher_rows)


def fetch_odds_events(api_key: str, sport_key: str = SPORT_KEY, regions: str = ODDS_REGIONS,
                      markets: str = ODDS_MARKETS, odds_format: str = ODDS_FORMAT) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fetch current odds from The Odds API and return events + long outcome snapshots."""
    if not api_key:
        print("ODDS_API_KEY is blank; skipping odds.")
        return pd.DataFrame(), pd.DataFrame()
    url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds"
    params = {
        "apiKey": api_key,
        "regions": regions,
        "markets": markets,
        "oddsFormat": odds_format,
    }
    r = requests.get(url, params=params, timeout=60)
    print("Odds API status:", r.status_code, "remaining:", r.headers.get("x-requests-remaining"), "used:", r.headers.get("x-requests-used"))
    r.raise_for_status()
    data = r.json()
    fetched_at = pd.Timestamp.utcnow().isoformat()
    event_rows = []
    snap_rows = []
    for ev in data:
        event_id = ev.get("id")
        event_rows.append({
            "event_id": event_id,
            "sport_key": ev.get("sport_key"),
            "sport_title": ev.get("sport_title"),
            "commence_time_utc": ev.get("commence_time"),
            "home_team": ev.get("home_team"),
            "away_team": ev.get("away_team"),
            "home_team_norm": normalize_team_name(ev.get("home_team")),
            "away_team_norm": normalize_team_name(ev.get("away_team")),
            "fetched_at_utc": fetched_at,
        })
        for book in ev.get("bookmakers", []) or []:
            for market in book.get("markets", []) or []:
                mkey = market.get("key")
                for outcome in market.get("outcomes", []) or []:
                    snap_rows.append({
                        "fetched_at_utc": fetched_at,
                        "event_id": event_id,
                        "sport_key": ev.get("sport_key"),
                        "commence_time_utc": ev.get("commence_time"),
                        "home_team": ev.get("home_team"),
                        "away_team": ev.get("away_team"),
                        "home_team_norm": normalize_team_name(ev.get("home_team")),
                        "away_team_norm": normalize_team_name(ev.get("away_team")),
                        "bookmaker_key": book.get("key"),
                        "bookmaker_title": book.get("title"),
                        "market_key": mkey,
                        "market_last_update": market.get("last_update"),
                        "outcome_name": outcome.get("name"),
                        "outcome_name_norm": normalize_team_name(outcome.get("name")),
                        "outcome_price": outcome.get("price"),
                        "outcome_point": outcome.get("point"),
                    })
    events = pd.DataFrame(event_rows)
    snapshots = pd.DataFrame(snap_rows)
    for df in [events, snapshots]:
        if not df.empty:
            for c in ["commence_time_utc", "fetched_at_utc", "market_last_update"]:
                if c in df.columns:
                    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)
    return events, snapshots


def fetch_statcast_pybaseball(start_date: str, end_date: str, chunk_days: int = 7) -> pd.DataFrame:
    """Fetch pitch-level Statcast using pybaseball.statcast."""
    try:
        from pybaseball import statcast
    except Exception as exc:
        raise ImportError("pybaseball is required for Statcast pulls. Run `%pip install pybaseball`.") from exc

    frames = []
    for s, e in daterange_chunks(start_date, end_date, chunk_days):
        print(f"Fetching Statcast {s} to {e}")
        try:
            chunk = statcast(s.isoformat(), e.isoformat())
            if chunk is not None and not chunk.empty:
                frames.append(chunk)
                print("  rows", len(chunk))
            else:
                print("  empty chunk")
        except Exception as exc:
            print(f"  Statcast chunk failed {s} to {e}: {type(exc).__name__}: {exc}")
        time.sleep(API_SLEEP_SECONDS)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True, sort=False)
    out = out.drop_duplicates()
    return out

print("API clients ready")


## Fetch schedule

In [ ]:

if FETCH_SCHEDULE:
    games = fetch_mlb_schedule(START_DATE, END_DATE, game_type=GAME_TYPE, chunk_days=SCHEDULE_CHUNK_DAYS)
else:
    games = pd.DataFrame()

show_df("games", games)
print("completed rows", games["target_home_win"].notna().sum() if "target_home_win" in games.columns else 0)


## Fetch boxscores

In [ ]:

if FETCH_BOXSCORES and not games.empty:
    box_team_game, box_pitcher_game = fetch_boxscores_for_games_full(games, only_final=True, limit=None)
else:
    box_team_game = pd.DataFrame()
    box_pitcher_game = pd.DataFrame()

show_df("box_team_game", box_team_game)
show_df("box_pitcher_game", box_pitcher_game)
if not box_pitcher_game.empty and "is_starting_pitcher" in box_pitcher_game.columns:
    print("starter rows", int(box_pitcher_game["is_starting_pitcher"].sum()))


## Fetch current odds

In [ ]:

if FETCH_ODDS:
    odds_events, odds_snapshots = fetch_odds_events(ODDS_API_KEY)
else:
    odds_events = pd.DataFrame()
    odds_snapshots = pd.DataFrame()

show_df("odds_events", odds_events)
show_df("odds_snapshots", odds_snapshots)


## Fetch Statcast

In [ ]:

if FETCH_STATCAST:
    statcast_raw = fetch_statcast_pybaseball(START_DATE, END_DATE, chunk_days=STATCAST_CHUNK_DAYS)
else:
    statcast_raw = pd.DataFrame()

show_df("statcast_raw", statcast_raw)


## Repair timestamps and align team keys

In [ ]:

def repair_game_time_from_games(df: pd.DataFrame, games_df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Add official_date/game_datetime_utc to raw tables using game_pk lookup from schedule."""
    if df is None or df.empty:
        print(f"{name}: empty")
        return pd.DataFrame()

    d = df.copy()
    lookup = games_df[["game_pk", "official_date", "game_datetime_utc"]].drop_duplicates("game_pk").copy()
    lookup["game_pk"] = pd.to_numeric(lookup["game_pk"], errors="coerce")
    lookup["official_date"] = pd.to_datetime(lookup["official_date"], errors="coerce")
    lookup["game_datetime_utc"] = pd.to_datetime(lookup["game_datetime_utc"], errors="coerce", utc=True)

    d["game_pk"] = pd.to_numeric(d["game_pk"], errors="coerce")

    if "official_date" not in d.columns and "game_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["game_date"], errors="coerce")

    d = d.merge(
        lookup.rename(columns={
            "official_date": "_official_date_from_games",
            "game_datetime_utc": "_game_datetime_utc_from_games",
        }),
        on="game_pk",
        how="left",
    )

    if "official_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["official_date"], errors="coerce")
        d["official_date"] = d["official_date"].combine_first(d["_official_date_from_games"])
    else:
        d["official_date"] = d["_official_date_from_games"]

    if "game_datetime_utc" in d.columns:
        d["game_datetime_utc"] = pd.to_datetime(d["game_datetime_utc"], errors="coerce", utc=True)
        d["game_datetime_utc"] = d["game_datetime_utc"].combine_first(d["_game_datetime_utc_from_games"])
    else:
        d["game_datetime_utc"] = d["_game_datetime_utc_from_games"]

    d = d.drop(columns=["_official_date_from_games", "_game_datetime_utc_from_games"], errors="ignore")

    print(
        f"{name}: repaired timestamps; missing game_datetime_utc = "
        f"{d['game_datetime_utc'].isna().sum():,} of {len(d):,}"
    )
    return d


# Filter Statcast down to the game universe from the schedule. This removes spring-training or unmatched rows.
if not statcast_raw.empty and not games.empty and "game_pk" in statcast_raw.columns:
    games_pks = set(pd.to_numeric(games["game_pk"], errors="coerce").dropna().astype(int))
    before = len(statcast_raw)
    statcast_raw["game_pk"] = pd.to_numeric(statcast_raw["game_pk"], errors="coerce")
    statcast_raw = statcast_raw[statcast_raw["game_pk"].isin(games_pks)].copy()
    statcast_raw = repair_game_time_from_games(statcast_raw, games, "statcast_raw")
    after = len(statcast_raw)
    print(f"Statcast rows filtered to scheduled games: {before:,} -> {after:,}")

# Ensure game table team keys use the same abbreviation mapping as Statcast.
if not games.empty:
    games = games.copy()
    games["home_team_norm"] = games["home_team_name"].apply(normalize_team_name)
    games["away_team_norm"] = games["away_team_name"].apply(normalize_team_name)

# Optional overlap audit.
if not statcast_raw.empty and {"home_team", "away_team"}.issubset(statcast_raw.columns):
    sc_team_keys = sorted({normalize_team_name(x) for x in pd.concat([statcast_raw["home_team"], statcast_raw["away_team"]], ignore_index=True).dropna().unique()})
    game_team_keys = sorted(set(games["home_team_norm"].dropna().astype(str)) | set(games["away_team_norm"].dropna().astype(str)))
    print("Statcast keys not in games:", sorted(set(sc_team_keys) - set(game_team_keys)))
    print("Game keys not in Statcast:", sorted(set(game_team_keys) - set(sc_team_keys)))


## Inline feature engineering functions

In [ ]:
def coerce_numeric_cols(df: pd.DataFrame, skip: set[str] | None = None) -> pd.DataFrame:
    out = df.copy()
    skip = skip or set()
    for c in out.columns:
        if c in skip:
            continue
        if out[c].dtype == object:
            # Convert strings like '.321' or '1.234' when possible.
            converted = pd.to_numeric(out[c].astype(str).str.replace("%", "", regex=False), errors="ignore")
            out[c] = converted
    return out


def add_basic_game_outcome_features(games_df: pd.DataFrame) -> pd.DataFrame:
    g = games_df.copy()
    g["home_win"] = np.where(g["is_final"], (g["home_score"] > g["away_score"]).astype(float), np.nan)
    g["away_win"] = np.where(g["is_final"], (g["away_score"] > g["home_score"]).astype(float), np.nan)
    g["home_run_diff"] = np.where(g["is_final"], g["home_score"] - g["away_score"], np.nan)
    g["away_run_diff"] = np.where(g["is_final"], g["away_score"] - g["home_score"], np.nan)
    return g


def team_game_long_from_games(games_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in games_df.iterrows():
        if not bool(r.get("is_final")):
            continue
        for side in ["home", "away"]:
            opp = "away" if side == "home" else "home"
            runs_for = r.get(f"{side}_score")
            runs_against = r.get(f"{opp}_score")
            rows.append({
                "game_pk": r.get("game_pk"),
                "official_date": r.get("official_date"),
                "game_datetime_utc": r.get("game_datetime_utc"),
                "team_side": side,
                "team_id": r.get(f"{side}_team_id"),
                "team_name": r.get(f"{side}_team_name"),
                "team_norm": r.get(f"{side}_team_norm"),
                "opponent_team_id": r.get(f"{opp}_team_id"),
                "opponent_team_name": r.get(f"{opp}_team_name"),
                "runs_for": runs_for,
                "runs_against": runs_against,
                "win": float(runs_for > runs_against) if pd.notna(runs_for) and pd.notna(runs_against) else np.nan,
                "run_diff": runs_for - runs_against if pd.notna(runs_for) and pd.notna(runs_against) else np.nan,
            })
    return pd.DataFrame(rows)


def add_rolling_entity_features(long_df: pd.DataFrame, entity_col: str, value_cols: list[str], prefix: str,
                                windows: list[int] = [3, 5, 10, 20], season: bool = True) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame()
    d = long_df.copy().sort_values([entity_col, "official_date", "game_datetime_utc", "game_pk"])
    out = d[["game_pk", entity_col]].copy()
    for col in value_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        shifted = d.groupby(entity_col)[col].shift(1)
        if season:
            out[f"{prefix}_{col}_season_to_date"] = shifted.groupby(d[entity_col]).expanding(min_periods=1).mean().reset_index(level=0, drop=True)
        for w in windows:
            out[f"{prefix}_{col}_last{w}"] = shifted.groupby(d[entity_col]).rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
    return pd.concat([d[["game_pk", entity_col, "official_date", "game_datetime_utc"]].reset_index(drop=True), out.drop(columns=["game_pk", entity_col]).reset_index(drop=True)], axis=1)


def compute_elo_features(games_df: pd.DataFrame, k: float = 20.0, home_adv: float = 35.0, base_elo: float = 1500.0) -> pd.DataFrame:
    g = games_df.copy().sort_values(["official_date", "game_datetime_utc", "game_pk"])
    ratings: dict[int, float] = {}
    rows = []
    for _, r in g.iterrows():
        h = int(r["home_team_id"]) if pd.notna(r.get("home_team_id")) else None
        a = int(r["away_team_id"]) if pd.notna(r.get("away_team_id")) else None
        if h is None or a is None:
            continue
        rh = ratings.get(h, base_elo)
        ra = ratings.get(a, base_elo)
        ph = 1.0 / (1.0 + 10 ** (-((rh + home_adv) - ra) / 400.0))
        rows.append({
            "game_pk": r.get("game_pk"),
            "home_elo_pre": rh,
            "away_elo_pre": ra,
            "diff_elo_pre": rh - ra,
            "elo_home_win_prob": ph,
        })
        if bool(r.get("is_final")) and pd.notna(r.get("home_score")) and pd.notna(r.get("away_score")):
            outcome = 1.0 if r["home_score"] > r["away_score"] else 0.0
            change = k * (outcome - ph)
            ratings[h] = rh + change
            ratings[a] = ra - change
    return pd.DataFrame(rows)

print("Basic rolling/ELO functions ready")


In [ ]:

def pitch_family(pt: Any) -> str:
    if pt is None or pd.isna(pt):
        return "unknown"
    pt = str(pt).upper()
    fast = {"FF", "SI", "FC", "FA", "FS"}
    breaking = {"SL", "CU", "KC", "SV", "ST"}
    offspeed = {"CH", "FS", "FO", "SC"}
    if pt in fast:
        return "fastball"
    if pt in breaking:
        return "breaking"
    if pt in offspeed:
        return "offspeed"
    return "other"


def entropy_from_counts(counts: pd.Series) -> float:
    values = counts[counts > 0].astype(float)
    if values.sum() <= 0:
        return np.nan
    p = values / values.sum()
    return float(-(p * np.log(p)).sum())

def prepare_statcast_pitch_level(sc: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare raw pybaseball Statcast rows without creating duplicate official_date columns.

    repair_game_time_from_games() may already add official_date and game_datetime_utc
    while leaving pybaseball's game_date in place. Do NOT rename game_date -> official_date
    when official_date already exists.
    """
    if sc is None or sc.empty:
        return pd.DataFrame()

    d = sc.copy()

    # Defensive cleanup in case a prior notebook run created duplicate labels.
    if not d.columns.is_unique:
        out = pd.DataFrame(index=d.index)

        for col in pd.Index(d.columns).unique():
            same = d.loc[:, d.columns == col]

            if same.shape[1] == 1:
                out[col] = same.iloc[:, 0]
            else:
                s = same.iloc[:, 0]
                for j in range(1, same.shape[1]):
                    s = s.combine_first(same.iloc[:, j])
                out[col] = s

        d = out

    # Use official_date if present; otherwise derive it from pybaseball game_date.
    if "official_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["official_date"], errors="coerce")
    elif "game_date" in d.columns:
        d["official_date"] = pd.to_datetime(d["game_date"], errors="coerce")

    # Keep UTC start time from MLB schedule lookup if available.
    if "game_datetime_utc" in d.columns:
        d["game_datetime_utc"] = pd.to_datetime(
            d["game_datetime_utc"],
            errors="coerce",
            utc=True,
        )
    elif "official_date" in d.columns:
        d["game_datetime_utc"] = pd.to_datetime(
            d["official_date"],
            errors="coerce",
            utc=True,
        )

    if "game_pk" in d.columns:
        d["game_pk"] = pd.to_numeric(d["game_pk"], errors="coerce")

    for c in [
        "release_speed",
        "release_spin_rate",
        "release_extension",
        "launch_speed",
        "launch_angle",
        "estimated_woba_using_speedangle",
        "woba_value",
        "estimated_ba_using_speedangle",
    ]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    if {"inning_topbot", "home_team", "away_team"}.issubset(d.columns):
        d["bat_team"] = np.where(
            d["inning_topbot"].astype(str).str.lower().eq("top"),
            d["away_team"],
            d["home_team"],
        )

        d["pitch_team"] = np.where(
            d["inning_topbot"].astype(str).str.lower().eq("top"),
            d["home_team"],
            d["away_team"],
        )

    d["bat_team_norm"] = d.get(
        "bat_team",
        pd.Series(index=d.index, dtype=object),
    ).apply(normalize_team_name)

    d["pitch_team_norm"] = d.get(
        "pitch_team",
        pd.Series(index=d.index, dtype=object),
    ).apply(normalize_team_name)

    desc = d.get("description", pd.Series("", index=d.index)).fillna("").astype(str)
    events = d.get("events", pd.Series("", index=d.index)).fillna("").astype(str)

    d["is_pa_event"] = events.ne("")
    d["is_strikeout"] = events.str.contains("strikeout", case=False, na=False)
    d["is_walk"] = (
        events.str.contains("walk", case=False, na=False)
        & ~events.str.contains("intent", case=False, na=False)
    )
    d["is_home_run"] = events.str.contains("home_run", case=False, na=False)

    d["is_batted_ball"] = d.get(
        "launch_speed",
        pd.Series(np.nan, index=d.index),
    ).notna()

    d["is_hard_hit"] = d.get(
        "launch_speed",
        pd.Series(np.nan, index=d.index),
    ).ge(95)

    d["is_sweetspot"] = d.get(
        "launch_angle",
        pd.Series(np.nan, index=d.index),
    ).between(8, 32)

    d["is_whiff"] = desc.isin(
        ["swinging_strike", "swinging_strike_blocked", "foul_tip"]
    )
    d["is_called_strike"] = desc.eq("called_strike")
    d["is_swing"] = desc.str.contains(
        "swing|foul|hit_into_play",
        case=False,
        regex=True,
        na=False,
    )

    d["pitch_family"] = d.get(
        "pitch_type",
        pd.Series("UNK", index=d.index),
    ).fillna("UNK").map(pitch_family)

    return d

In [ ]:
def rolling_team_features_from_games(games_df: pd.DataFrame) -> pd.DataFrame:
    long = team_game_long_from_games(games_df)
    if long.empty:
        return pd.DataFrame()
    value_cols = ["runs_for", "runs_against", "win", "run_diff"]
    return add_rolling_entity_features(long, "team_id", value_cols, "team")


def rolling_boxscore_features(box_df: pd.DataFrame) -> pd.DataFrame:
    if box_df.empty:
        return pd.DataFrame()
    d = coerce_numeric_cols(box_df, skip={"team_name", "team_norm", "opponent_team_name", "team_side"})
    value_cols = []
    for c in d.columns:
        if c.startswith("box_bat_") or c.startswith("box_pitch_") or c in ["runs_for", "runs_against"]:
            if pd.api.types.is_numeric_dtype(d[c]):
                value_cols.append(c)
    value_cols = value_cols[:80]  # keep lab manageable; remove cap if desired.
    return add_rolling_entity_features(d, "team_id", value_cols, "box")


def rolling_statcast_team_features(sc_team_game: pd.DataFrame) -> pd.DataFrame:
    if sc_team_game.empty:
        return pd.DataFrame()
    value_cols = [c for c in sc_team_game.columns if c.startswith("sc_")]
    return add_rolling_entity_features(sc_team_game, "team_norm", value_cols, "team_off")


def rolling_statcast_pitcher_features(sc_pitcher_game: pd.DataFrame) -> pd.DataFrame:
    if sc_pitcher_game.empty:
        return pd.DataFrame()
    value_cols = [c for c in sc_pitcher_game.columns if c.startswith("sc_")]
    return add_rolling_entity_features(sc_pitcher_game, "pitcher_id", value_cols, "starter_statcast")


def rolling_pitchmix_team_features(team_pt: pd.DataFrame) -> pd.DataFrame:
    if team_pt.empty:
        return pd.DataFrame()
    # Pivot pitch families wide per game/team.
    piv = team_pt.pivot_table(index=["game_pk", "official_date", "team_norm"], columns="pitch_family", values=["pitches", "avg_ev", "woba", "whiff_rate", "hard_hit_rate"], aggfunc="mean")
    piv.columns = [f"pt_{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    value_cols = [c for c in piv.columns if c.startswith("pt_")]
    return add_rolling_entity_features(piv, "team_norm", value_cols, "team_pitchmix")

print("Rolling feature functions ready")


In [ ]:
def merge_home_away_team_features(base: pd.DataFrame, feat: pd.DataFrame, entity_col: str, prefix: str, id_home_col: str, id_away_col: str) -> pd.DataFrame:
    if feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in feat.columns if c not in {"game_pk", entity_col, "official_date", "game_datetime_utc"}]
    latest = feat[["game_pk", entity_col] + feature_cols].drop_duplicates(["game_pk", entity_col], keep="last")

    home = latest.rename(columns={entity_col: id_home_col, **{c: f"home_{prefix}_{c}" for c in feature_cols}})
    away = latest.rename(columns={entity_col: id_away_col, **{c: f"away_{prefix}_{c}" for c in feature_cols}})

    d = d.merge(home, on=["game_pk", id_home_col], how="left")
    d = d.merge(away, on=["game_pk", id_away_col], how="left")

    for c in feature_cols:
        hc = f"home_{prefix}_{c}"
        ac = f"away_{prefix}_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_{prefix}_{c}"] = d[hc] - d[ac]
    return d


def attach_latest_h2h_odds_features(games_df: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    d = games_df.copy()
    if odds_df.empty:
        return d
    h2h = odds_df[odds_df["market_key"].astype(str).str.lower().isin(["h2h", "moneyline"])]
    if h2h.empty:
        return d
    event_level = []
    for event_id, ev in h2h.groupby("event_id"):
        ev0 = ev.iloc[0]
        home_norm = ev0["home_team_norm"]
        away_norm = ev0["away_team_norm"]
        home_prices = pd.to_numeric(ev.loc[ev["outcome_name_norm"].eq(home_norm), "outcome_price"], errors="coerce").dropna()
        away_prices = pd.to_numeric(ev.loc[ev["outcome_name_norm"].eq(away_norm), "outcome_price"], errors="coerce").dropna()
        if home_prices.empty or away_prices.empty:
            continue
        hp = float(home_prices.median())
        ap = float(away_prices.median())
        home_nv, away_nv = no_vig_two_way_prob(hp, ap)
        event_level.append({
            "event_id": event_id,
            "odds_commence_time_utc": ev0["commence_time_utc"],
            "home_team_norm": home_norm,
            "away_team_norm": away_norm,
            "home_moneyline_median": hp,
            "away_moneyline_median": ap,
            "market_home_no_vig_prob": home_nv,
            "market_away_no_vig_prob": away_nv,
        })
    evdf = pd.DataFrame(event_level)
    if evdf.empty:
        return d
    # Match by team names and nearest start time within 3 hours.
    rows = []
    for idx, g in d.iterrows():
        cand = evdf[(evdf["home_team_norm"].eq(g.get("home_team_norm"))) & (evdf["away_team_norm"].eq(g.get("away_team_norm")))].copy()
        if cand.empty:
            rows.append({})
            continue
        cand["dt_min"] = (cand["odds_commence_time_utc"] - g["game_datetime_utc"]).abs().dt.total_seconds() / 60.0
        cand = cand.sort_values("dt_min")
        best = cand.iloc[0]
        if best["dt_min"] <= 180:
            rows.append(best.drop(labels=["home_team_norm", "away_team_norm"]).to_dict())
        else:
            rows.append({})
    attach = pd.DataFrame(rows)
    d = pd.concat([d.reset_index(drop=True), attach.reset_index(drop=True)], axis=1)
    d["has_market_odds"] = d.get("home_moneyline_median", pd.Series(np.nan, index=d.index)).notna().astype(int)
    return d


def build_game_feature_frame(games_df: pd.DataFrame, box_df: pd.DataFrame, statcast_raw_df: pd.DataFrame, odds_df: pd.DataFrame) -> pd.DataFrame:
    base = add_basic_game_outcome_features(games_df).copy()
    elo = compute_elo_features(base)
    if not elo.empty:
        base = base.merge(elo, on="game_pk", how="left")

    team_roll = rolling_team_features_from_games(base)
    base = merge_home_away_team_features(base, team_roll, "team_id", "team", "home_team_id", "away_team_id")

    box_roll = rolling_boxscore_features(box_df)
    base = merge_home_away_team_features(base, box_roll, "team_id", "box", "home_team_id", "away_team_id")

    if not statcast_raw_df.empty:
        sc_team = aggregate_statcast_team_game(statcast_raw_df)
        sc_pitcher = aggregate_statcast_pitcher_game(statcast_raw_df)
        team_pt, pit_pt = aggregate_statcast_pitch_type(statcast_raw_df)
        sc_team_roll = rolling_statcast_team_features(sc_team)
        base = merge_home_away_team_features(base, sc_team_roll, "team_norm", "team_sc", "home_team_norm", "away_team_norm")
        pt_roll = rolling_pitchmix_team_features(team_pt)
        base = merge_home_away_team_features(base, pt_roll, "team_norm", "team_pitchmix", "home_team_norm", "away_team_norm")
        sp_roll = rolling_statcast_pitcher_features(sc_pitcher)
        base = merge_home_away_starter_features(base, sp_roll)
    else:
        sc_team = sc_pitcher = team_pt = pit_pt = pd.DataFrame()

    base = attach_latest_h2h_odds_features(base, odds_df)
    return base


def merge_home_away_starter_features(base: pd.DataFrame, sp_feat: pd.DataFrame) -> pd.DataFrame:
    if sp_feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in sp_feat.columns if c not in {"game_pk", "pitcher_id", "official_date", "game_datetime_utc"}]
    latest = sp_feat[["game_pk", "pitcher_id"] + feature_cols].drop_duplicates(["game_pk", "pitcher_id"], keep="last")
    home = latest.rename(columns={"pitcher_id": "home_probable_pitcher_id", **{c: f"home_starter_{c}" for c in feature_cols}})
    away = latest.rename(columns={"pitcher_id": "away_probable_pitcher_id", **{c: f"away_starter_{c}" for c in feature_cols}})
    d = d.merge(home, on=["game_pk", "home_probable_pitcher_id"], how="left")
    d = d.merge(away, on=["game_pk", "away_probable_pitcher_id"], how="left")
    for c in feature_cols:
        hc = f"home_starter_{c}"
        ac = f"away_starter_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_starter_{c}"] = d[hc] - d[ac]
    return d

print("Game feature builder ready")


In [ ]:
# -----------------------------------------------------------------------------
# PATCH: preserve game_datetime_utc through Statcast aggregations
# -----------------------------------------------------------------------------
# The raw Statcast table has game_datetime_utc after repair, but the earlier
# aggregate_statcast_* functions grouped only by game_pk/official_date/team.
# That dropped game_datetime_utc, and the rolling feature function then failed
# with KeyError: 'game_datetime_utc'. These definitions override the prior ones.

def _ensure_game_time_columns(d: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    if d is None or d.empty:
        return pd.DataFrame() if d is None else d.copy()
    out = d.copy()
    if not out.columns.is_unique:
        dedup = pd.DataFrame(index=out.index)
        for col in pd.Index(out.columns).unique():
            same = out.loc[:, out.columns == col]
            if same.shape[1] == 1:
                dedup[col] = same.iloc[:, 0]
            else:
                s = same.iloc[:, 0]
                for j in range(1, same.shape[1]):
                    s = s.combine_first(same.iloc[:, j])
                dedup[col] = s
        out = dedup

    if "official_date" in out.columns:
        out["official_date"] = pd.to_datetime(out["official_date"], errors="coerce")
    elif "game_date" in out.columns:
        out["official_date"] = pd.to_datetime(out["game_date"], errors="coerce")

    if "game_datetime_utc" in out.columns:
        out["game_datetime_utc"] = pd.to_datetime(out["game_datetime_utc"], errors="coerce", utc=True)
    elif "official_date" in out.columns:
        # Fallback only. It preserves chronological ordering by date when exact first pitch is unavailable.
        out["game_datetime_utc"] = pd.to_datetime(out["official_date"], errors="coerce", utc=True)
    else:
        raise KeyError(f"{name} is missing both game_datetime_utc and official_date. Columns: {list(out.columns)}")

    if "game_pk" in out.columns:
        out["game_pk"] = pd.to_numeric(out["game_pk"], errors="coerce")
    return out


def add_rolling_entity_features(long_df: pd.DataFrame, entity_col: str, value_cols: list[str], prefix: str,
                                windows: list[int] = [3, 5, 10, 20], season: bool = True) -> pd.DataFrame:
    if long_df is None or long_df.empty:
        return pd.DataFrame()
    d = _ensure_game_time_columns(long_df, f"rolling_input_{prefix}")
    if entity_col not in d.columns:
        raise KeyError(f"{prefix}: missing entity column {entity_col}. Columns: {list(d.columns)}")
    sort_cols = [entity_col, "official_date", "game_datetime_utc", "game_pk"]
    d = d.sort_values(sort_cols).reset_index(drop=True)

    out = d[["game_pk", entity_col]].copy()
    for col in value_cols:
        if col not in d.columns:
            continue
        d[col] = pd.to_numeric(d[col], errors="coerce")
        shifted = d.groupby(entity_col)[col].shift(1)
        if season:
            out[f"{prefix}_{col}_season_to_date"] = (
                shifted.groupby(d[entity_col])
                .expanding(min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
        for w in windows:
            out[f"{prefix}_{col}_last{w}"] = (
                shifted.groupby(d[entity_col])
                .rolling(w, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )

    return pd.concat(
        [
            d[["game_pk", entity_col, "official_date", "game_datetime_utc"]].reset_index(drop=True),
            out.drop(columns=["game_pk", entity_col], errors="ignore").reset_index(drop=True),
        ],
        axis=1,
    )


def aggregate_statcast_team_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc is None or sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    d = _ensure_game_time_columns(d, "prepared_statcast_team")
    group_cols = ["game_pk", "official_date", "game_datetime_utc", "bat_team_norm"]
    agg = d.groupby(group_cols).agg(
        sc_pitches_seen=("game_pk", "size"),
        sc_pa=("is_pa_event", "sum"),
        sc_avg_ev=("launch_speed", "mean"),
        sc_max_ev=("launch_speed", "max"),
        sc_avg_la=("launch_angle", "mean"),
        sc_hard_hit_rate=("is_hard_hit", "mean"),
        sc_sweetspot_rate=("is_sweetspot", "mean"),
        sc_xwoba_contact=("estimated_woba_using_speedangle", "mean"),
        sc_woba=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
        sc_swing_rate=("is_swing", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})

    csw = (
        d.assign(csw=d["is_called_strike"] | d["is_whiff"])
        .groupby(group_cols)["csw"]
        .mean()
        .reset_index(name="sc_csw_rate")
        .rename(columns={"bat_team_norm": "team_norm"})
    )
    agg = agg.merge(csw, on=["game_pk", "official_date", "game_datetime_utc", "team_norm"], how="left")
    return agg


def aggregate_statcast_pitcher_game(sc: pd.DataFrame) -> pd.DataFrame:
    if sc is None or sc.empty:
        return pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    d = _ensure_game_time_columns(d, "prepared_statcast_pitcher")
    if "pitcher" not in d.columns:
        return pd.DataFrame()
    group_cols = ["game_pk", "official_date", "game_datetime_utc", "pitcher"]
    agg = d.groupby(group_cols).agg(
        sc_pitches=("game_pk", "size"),
        sc_avg_velo=("release_speed", "mean"),
        sc_max_velo=("release_speed", "max"),
        sc_avg_spin=("release_spin_rate", "mean"),
        sc_avg_extension=("release_extension", "mean"),
        sc_whiff_rate=("is_whiff", "mean"),
        sc_called_strike_rate=("is_called_strike", "mean"),
        sc_swing_rate=("is_swing", "mean"),
        sc_hard_hit_allowed=("is_hard_hit", "mean"),
        sc_xwoba_allowed=("estimated_woba_using_speedangle", "mean"),
        sc_woba_allowed=("woba_value", "mean"),
        sc_k_rate=("is_strikeout", "mean"),
        sc_bb_rate=("is_walk", "mean"),
        sc_hr_rate=("is_home_run", "mean"),
    ).reset_index().rename(columns={"pitcher": "pitcher_id"})

    ent = (
        d.groupby(group_cols + ["pitch_family"]).size()
        .reset_index(name="n")
        .groupby(group_cols)["n"]
        .apply(entropy_from_counts)
        .reset_index(name="sc_pitchmix_entropy")
        .rename(columns={"pitcher": "pitcher_id"})
    )
    agg = agg.merge(ent, on=["game_pk", "official_date", "game_datetime_utc", "pitcher_id"], how="left")
    return agg


def aggregate_statcast_pitch_type(sc: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if sc is None or sc.empty:
        return pd.DataFrame(), pd.DataFrame()
    d = prepare_statcast_pitch_level(sc)
    d = _ensure_game_time_columns(d, "prepared_statcast_pitch_type")

    team_pt = d.groupby(["game_pk", "official_date", "game_datetime_utc", "bat_team_norm", "pitch_family"]).agg(
        pitches=("game_pk", "size"),
        avg_ev=("launch_speed", "mean"),
        woba=("woba_value", "mean"),
        whiff_rate=("is_whiff", "mean"),
        hard_hit_rate=("is_hard_hit", "mean"),
    ).reset_index().rename(columns={"bat_team_norm": "team_norm"})

    if "pitcher" in d.columns:
        pit_pt = d.groupby(["game_pk", "official_date", "game_datetime_utc", "pitcher", "pitch_family"]).agg(
            pitches=("game_pk", "size"),
            avg_velo=("release_speed", "mean"),
            whiff_rate=("is_whiff", "mean"),
            usage=("game_pk", "size"),
        ).reset_index().rename(columns={"pitcher": "pitcher_id"})
    else:
        pit_pt = pd.DataFrame()
    return team_pt, pit_pt


def rolling_pitchmix_team_features(team_pt: pd.DataFrame) -> pd.DataFrame:
    if team_pt is None or team_pt.empty:
        return pd.DataFrame()
    d = _ensure_game_time_columns(team_pt, "team_pitchmix")
    piv = d.pivot_table(
        index=["game_pk", "official_date", "game_datetime_utc", "team_norm"],
        columns="pitch_family",
        values=["pitches", "avg_ev", "woba", "whiff_rate", "hard_hit_rate"],
        aggfunc="mean",
    )
    piv.columns = [f"pt_{a}_{b}" for a, b in piv.columns]
    piv = piv.reset_index()
    value_cols = [c for c in piv.columns if c.startswith("pt_")]
    return add_rolling_entity_features(piv, "team_norm", value_cols, "team_pitchmix")

print("Patched Statcast aggregation/rolling functions to preserve game_datetime_utc")


In [ ]:

def rolling_starter_boxscore_features(box_pitcher_df: pd.DataFrame, windows: list[int] = [1, 3, 5, 10]) -> pd.DataFrame:
    """Leakage-safe rolling starter ERA/WHIP/K/9/BB/9/HR/9 using prior starts only."""
    if box_pitcher_df is None or box_pitcher_df.empty:
        return pd.DataFrame()

    d = box_pitcher_df.copy()
    d = _ensure_game_time_columns(d, "box_pitcher_game")
    if "is_starting_pitcher" in d.columns:
        d = d[pd.to_numeric(d["is_starting_pitcher"], errors="coerce").fillna(0).astype(int).eq(1)].copy()
    if d.empty:
        return pd.DataFrame()

    for c in ["pitcher_id", "p_ip_float", "p_hits", "p_earned_runs", "p_base_on_balls", "p_strikeouts", "p_home_runs", "p_pitches_thrown"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.sort_values(["pitcher_id", "official_date", "game_datetime_utc", "game_pk"]).reset_index(drop=True)
    out = d[["game_pk", "pitcher_id", "official_date", "game_datetime_utc"]].copy()

    # Shift each raw count so the current game does not leak into its own features.
    grouped = d.groupby("pitcher_id", group_keys=False)
    shifted = pd.DataFrame(index=d.index)
    for c in ["p_ip_float", "p_hits", "p_earned_runs", "p_base_on_balls", "p_strikeouts", "p_home_runs", "p_pitches_thrown"]:
        shifted[c] = grouped[c].shift(1)

    for w in windows:
        label = "last_outing" if w == 1 else f"rolling{w}"
        sums = {}
        for c in shifted.columns:
            sums[c] = shifted.groupby(d["pitcher_id"])[c].rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
        ip = sums["p_ip_float"].replace(0, np.nan)
        out[f"starter_box_{label}_ip"] = ip
        out[f"starter_box_{label}_era"] = 9.0 * sums["p_earned_runs"] / ip
        out[f"starter_box_{label}_whip"] = (sums["p_hits"] + sums["p_base_on_balls"]) / ip
        out[f"starter_box_{label}_k_per_9"] = 9.0 * sums["p_strikeouts"] / ip
        out[f"starter_box_{label}_bb_per_9"] = 9.0 * sums["p_base_on_balls"] / ip
        out[f"starter_box_{label}_hr_per_9"] = 9.0 * sums["p_home_runs"] / ip
        out[f"starter_box_{label}_avg_pitches"] = shifted.groupby(d["pitcher_id"])["p_pitches_thrown"].rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)

    return out


def merge_home_away_starter_box_features(base: pd.DataFrame, starter_feat: pd.DataFrame) -> pd.DataFrame:
    if starter_feat is None or starter_feat.empty:
        return base
    d = base.copy()
    feature_cols = [c for c in starter_feat.columns if c not in {"game_pk", "pitcher_id", "official_date", "game_datetime_utc"}]
    latest = starter_feat[["game_pk", "pitcher_id"] + feature_cols].drop_duplicates(["game_pk", "pitcher_id"], keep="last")
    home = latest.rename(columns={"pitcher_id": "home_probable_pitcher_id", **{c: f"home_{c}" for c in feature_cols}})
    away = latest.rename(columns={"pitcher_id": "away_probable_pitcher_id", **{c: f"away_{c}" for c in feature_cols}})
    d = d.merge(home, on=["game_pk", "home_probable_pitcher_id"], how="left")
    d = d.merge(away, on=["game_pk", "away_probable_pitcher_id"], how="left")
    for c in feature_cols:
        hc = f"home_{c}"
        ac = f"away_{c}"
        if hc in d.columns and ac in d.columns:
            d[f"diff_{c}"] = d[hc] - d[ac]
    return d


def build_game_feature_frame(games_df: pd.DataFrame, box_df: pd.DataFrame, statcast_raw_df: pd.DataFrame, odds_df: pd.DataFrame,
                             box_pitcher_df: pd.DataFrame | None = None) -> pd.DataFrame:
    """One central feature build. Uses only inline notebook functions."""
    base = add_basic_game_outcome_features(games_df).copy()
    elo = compute_elo_features(base)
    if not elo.empty:
        base = base.merge(elo, on="game_pk", how="left")

    team_roll = rolling_team_features_from_games(base)
    base = merge_home_away_team_features(base, team_roll, "team_id", "team", "home_team_id", "away_team_id")

    box_roll = rolling_boxscore_features(box_df)
    base = merge_home_away_team_features(base, box_roll, "team_id", "box", "home_team_id", "away_team_id")

    if box_pitcher_df is not None and not box_pitcher_df.empty:
        starter_box_roll = rolling_starter_boxscore_features(box_pitcher_df)
        base = merge_home_away_starter_box_features(base, starter_box_roll)

    if statcast_raw_df is not None and not statcast_raw_df.empty:
        sc_team = aggregate_statcast_team_game(statcast_raw_df)
        sc_pitcher = aggregate_statcast_pitcher_game(statcast_raw_df)
        team_pt, pit_pt = aggregate_statcast_pitch_type(statcast_raw_df)
        sc_team_roll = rolling_statcast_team_features(sc_team)
        base = merge_home_away_team_features(base, sc_team_roll, "team_norm", "team_sc", "home_team_norm", "away_team_norm")
        pt_roll = rolling_pitchmix_team_features(team_pt)
        base = merge_home_away_team_features(base, pt_roll, "team_norm", "team_pitchmix", "home_team_norm", "away_team_norm")
        sp_roll = rolling_statcast_pitcher_features(sc_pitcher)
        base = merge_home_away_starter_features(base, sp_roll)

    base = attach_latest_h2h_odds_features(base, odds_df)
    return base

print("Starter boxscore rolling features and final feature builder ready")


## Build the final feature dataframe

In [ ]:

features = build_game_feature_frame(
    games,
    box_team_game,
    statcast_raw,
    odds_snapshots,
    box_pitcher_game,
)

print("features", features.shape)
if "official_date" in features.columns:
    print("date range", features["official_date"].min(), features["official_date"].max())
if "target_home_win" in features.columns:
    print("completed rows", int(features["target_home_win"].notna().sum()))

# Drop columns that are 100% missing. These are often sparse pitch-family artifacts.
all_null_cols = features.columns[features.isna().all()].tolist()
print("100% missing columns", len(all_null_cols))
if all_null_cols:
    display(pd.DataFrame({"column": all_null_cols}).head(50))
    features = features.drop(columns=all_null_cols)
    print("features after all-null drop", features.shape)

display(features.head())


## Data quality reports

In [ ]:

# Lightweight data quality reports for local inspection.
numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()

feature_inventory = pd.DataFrame({
    "column": features.columns,
    "dtype": [str(features[c].dtype) for c in features.columns],
    "missing_rate": [float(features[c].isna().mean()) for c in features.columns],
    "n_unique": [int(features[c].nunique(dropna=True)) for c in features.columns],
})

def family_for_col(c: str) -> str:
    c = c.lower()
    if "starter_box" in c:
        return "starter_boxscore"
    if "starter_statcast" in c:
        return "starter_statcast"
    if "pitchmix" in c:
        return "pitchmix"
    if "team_sc" in c or "statcast" in c:
        return "statcast_team"
    if "box" in c:
        return "boxscore_team"
    if "elo" in c:
        return "elo"
    if "team_team" in c:
        return "team_form"
    if "moneyline" in c or "market" in c or "odds" in c:
        return "odds"
    if c.startswith("target") or c in {"home_score", "away_score"}:
        return "target_or_result"
    return "other"

feature_inventory["family"] = feature_inventory["column"].map(family_for_col)
feature_inventory = feature_inventory.sort_values(["family", "missing_rate", "column"])

family_summary = (
    feature_inventory
    .groupby("family")
    .agg(
        n_columns=("column", "count"),
        avg_missing=("missing_rate", "mean"),
        max_missing=("missing_rate", "max"),
        median_unique=("n_unique", "median"),
    )
    .reset_index()
    .sort_values("n_columns", ascending=False)
)

display(family_summary)
display(feature_inventory.head(30))


## Save parquet/CSV outputs

In [ ]:

# Save the raw and processed outputs into a portable local package.
# In Colab, the final cell will trigger browser download. In Vertex/Jupyter, use the file link or file browser.

metadata = {
    "created_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "start_date": START_DATE,
    "end_date": END_DATE,
    "game_type": GAME_TYPE,
    "fetch_schedule": FETCH_SCHEDULE,
    "fetch_boxscores": FETCH_BOXSCORES,
    "fetch_odds": FETCH_ODDS,
    "fetch_statcast": FETCH_STATCAST,
    "features_shape": list(features.shape),
    "games_shape": list(games.shape),
    "box_team_game_shape": list(box_team_game.shape),
    "box_pitcher_game_shape": list(box_pitcher_game.shape),
    "odds_snapshots_shape": list(odds_snapshots.shape),
    "statcast_raw_shape": list(statcast_raw.shape),
}

# Main modeling table.
safe_to_parquet(features, PROCESSED_DIR / "mlb_game_features.parquet")

# Useful raw/intermediate tables. These let you re-engineer features locally without another API pull.
if EXPORT_RAW_TABLES:
    safe_to_parquet(games, RAW_DIR / "mlb_games.parquet")
    safe_to_parquet(box_team_game, RAW_DIR / "mlb_box_team_game.parquet")
    safe_to_parquet(box_pitcher_game, RAW_DIR / "mlb_box_pitcher_game.parquet")
    safe_to_parquet(odds_events, RAW_DIR / "mlb_odds_events.parquet")
    safe_to_parquet(odds_snapshots, RAW_DIR / "mlb_odds_snapshots.parquet")

if EXPORT_RAW_STATCAST and not statcast_raw.empty:
    safe_to_parquet(statcast_raw, RAW_DIR / "mlb_statcast_raw.parquet")
else:
    # Save a small schema/sample so you know what was available without making a huge zip.
    if not statcast_raw.empty:
        statcast_raw.head(1000).to_parquet(RAW_DIR / "mlb_statcast_raw_sample_1000.parquet", index=False)
        pd.DataFrame({"column": statcast_raw.columns}).to_csv(RAW_DIR / "mlb_statcast_raw_columns.csv", index=False)

feature_inventory.to_csv(REPORTS_DIR / "feature_inventory.csv", index=False)
family_summary.to_csv(REPORTS_DIR / "feature_family_summary.csv", index=False)

if EXPORT_CSV_SAMPLE:
    features.head(5000).to_csv(PROCESSED_DIR / "mlb_game_features_sample_5000.csv", index=False)

with open(EXPORT_ROOT / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved export files under:", EXPORT_ROOT)
for p in sorted(EXPORT_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(EXPORT_ROOT)}  {p.stat().st_size / 1024 / 1024:.2f} MB")


## Zip and download to your local PC

In [ ]:

if MAKE_ZIP:
    zip_path = PROJECT_ROOT / ZIP_NAME
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=EXPORT_ROOT)
    print("Created:", zip_path)
    print(f"Size: {zip_path.stat().st_size / 1024 / 1024:.2f} MB")

    # Colab browser download.
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass

    # Jupyter/Vertex link. In Vertex AI Workbench you can also right-click in the file browser and download.
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(zip_path)))
    except Exception:
        pass
else:
    print("MAKE_ZIP is False. Files are saved under", EXPORT_ROOT)


## Local reload example

In [ ]:

# Local reload example. Use this on your PC after downloading/unzipping the export.
# Adjust LOCAL_EXPORT_ROOT to the folder where you unzipped the package.

LOCAL_EXPORT_ROOT = Path("mlb_data_export")
LOCAL_FEATURES_PATH = LOCAL_EXPORT_ROOT / "processed" / "mlb_game_features.parquet"

if LOCAL_FEATURES_PATH.exists():
    local_features = pd.read_parquet(LOCAL_FEATURES_PATH)
    print("local_features", local_features.shape)
    display(local_features.head())
else:
    print("Set LOCAL_EXPORT_ROOT to your unzipped folder, then rerun this cell.")
